In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import numpy as np
import pandas as pd
import networkx as nx
import torch
import networkx as nx
import pandas as pd
from torch_geometric.data import Data
from torch_geometric.utils import from_networkx
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
import torch.nn as nn
import warnings
warnings.filterwarnings("ignore")
# 设置样式（避免中文乱码）
plt.rcParams['font.family'] = ['DejaVu Sans', 'Arial', 'sans-serif']
sns.set_style("whitegrid")

d:\Tool\Anaconda\envs\ponzi_e\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 请将路径替换为您的实际文件路径
FILE_PATH = 'E:\\ponzi\\data\\2PonziContract_20220701.csv'
ADDRESS_COLUMN = 'address'
LABEL_COLUMN = 'label'

def clean_and_analyze_data(df: pd.DataFrame, addr_col: str, label_col: str) -> pd.DataFrame:
    """
    分析并清理 DataFrame，剔除标签不一致和重复的地址记录。
    
    Args:
        df: 原始数据 DataFrame.
        addr_col: 地址列名.
        label_col: 标签列名.

    Returns:
        pd.DataFrame: 清理后的 DataFrame，保证地址唯一且标签明确。
    """
    original_len = len(df)
    
    # --- 1. 识别标签不一致的地址 (核心清理步骤) ---
    
    # 统计每个地址有多少个不同的标签值
    label_counts = df.groupby(addr_col)[label_col].nunique()
    inconsistent_addresses = label_counts[label_counts > 1].index.tolist()
    
    # 移除所有记录 associated with inconsistent addresses
    df_step1 = df[~df[addr_col].isin(inconsistent_addresses)].copy()
    
    removed_inconsistent_count = len(inconsistent_addresses)
    removed_inconsistent_rows = original_len - len(df_step1)

    print("\n--- 数据清理报告 ---")
    print(f"原始数据行数: {original_len}")
    
    if removed_inconsistent_count > 0:
        print(f"❌ 1. 移除 '标签不一致' 的地址行数: {removed_inconsistent_rows} (涉及 {removed_inconsistent_count} 个地址)")
        print(f"   这些地址被完全移除以避免标签歧义。")
    else:
        print("✅ 1. 未发现标签不一致的地址，无需移除。")

    # --- 2. 移除一致标签的重复记录 ---
    
    # 对于剩余的地址 (其标签已保证一致)，移除重复行，只保留第一条
    df_cleaned = df_step1.drop_duplicates(subset=[addr_col], keep='first').copy()
    
    removed_consistent_duplicate_rows = len(df_step1) - len(df_cleaned)
    
    if removed_consistent_duplicate_rows > 0:
        print(f"♻️ 2. 移除 '标签一致但重复' 的地址行数: {removed_consistent_duplicate_rows}")
    else:
        print("✅ 2. 未发现标签一致的重复记录。")
        
    print(f"--------------------------------------------------")
    print(f"清理后的最终行数: {len(df_cleaned)}")
    print(f"最终唯一地址数: {len(df_cleaned[addr_col].unique())}")
    
    return df_cleaned


def analyze_contract_data(file_path: str, addr_col: str, label_col: str):
    """
    主分析函数，读取文件并进行清理。
    """
    try:
        # 1. 读取数据
        df = pd.read_csv(file_path)
        print(f"成功读取文件：{file_path}")
    except FileNotFoundError:
        print(f"错误：文件未找到，请检查路径：{file_path}")
        return
    except Exception as e:
        print(f"读取文件时发生错误: {e}")
        return

    # 2. 清理和分析数据
    cleaned_df = clean_and_analyze_data(df, addr_col, label_col)
    
    return cleaned_df

In [3]:
cleaned_data = analyze_contract_data(FILE_PATH, ADDRESS_COLUMN, LABEL_COLUMN)
ponzi_contract = cleaned_data

成功读取文件：E:\ponzi\data\2PonziContract_20220701.csv

--- 数据清理报告 ---
原始数据行数: 6498
❌ 1. 移除 '标签不一致' 的地址行数: 30 (涉及 15 个地址)
   这些地址被完全移除以避免标签歧义。
✅ 2. 未发现标签一致的重复记录。
--------------------------------------------------
清理后的最终行数: 6468
最终唯一地址数: 6468


In [4]:
## 我们的构图（EthTrace Graph）方式：抽象式理解将合约操作码视为一段Trace Chain过程 PUSH PUSH1 DUP …，每笔Chain都是有时间顺序的（严格按照顺序执行的）。
# 思考：我们为什么要考虑这种构图方式？
# 回答：比简单的一维序列更容易进行复杂的分析；将智能合约从静态的代码文本提升为动态的执行模型，并利用图来分析其复杂的时序和依赖关系。
def construct_EthTrace_graph(ponzi_contract, addr):
    # 1. 提取字符串并分割 (保持不变)
    opcode = ponzi_contract[ponzi_contract['address'] == addr]['opcode'].values[0]
    opcode_lines = opcode.split('\n')
    opcode_sequence_str = opcode_lines[0]

    # 2. 分割操作码字符串为列表 (保持不变)
    opcode_list = [op for op in opcode_sequence_str.split(' ') if op]
    
    # 3. 构建边列表 (在构建时直接进行映射)
    trace_data = []

    for i in range(len(opcode_list) - 1):
        from_op_str = opcode_list[i]
        to_op_str = opcode_list[i + 1]
        
        # timestep 从 0 开始，按 1 增长
        position = i + 1  
        
        # 保存 ID 而不是字符串
        trace_data.append((from_op_str, to_op_str, position))
    
    # 4. 将数据转换为 Pandas DataFrame
    TraceChain_df = pd.DataFrame(trace_data, columns=['from_opcode_id', 'to_opcode_id', 'position'])
    
    # ================= 创建操作码类型的one-hot编码 =======================
    # 获取所有唯一的操作码类型
    all_opcodes = pd.unique(TraceChain_df[['from_opcode_id', 'to_opcode_id']].values.ravel('K'))
    
    # 为每个操作码创建one-hot编码列
    for opcode in all_opcodes:
        # 创建列名
        col_name = f'opcode_{opcode}'
        # 如果from_opcode_id或to_opcode_id等于当前操作码，则标记为1，否则为0
        TraceChain_df[col_name] = ((TraceChain_df['from_opcode_id'] == opcode) | 
                                  (TraceChain_df['to_opcode_id'] == opcode)).astype(int)
    
    # ================= 对所有账户进行编码 =======================
    # 提取唯一的地址
    unique_addresses = pd.unique(TraceChain_df[['from_opcode_id', 'to_opcode_id']].values.ravel('K'))
    # 创建地址编码映射
    address_encoding = {address: idx for idx, address in enumerate(unique_addresses)}

    # 替换 all_txn 中'from' 和 'to' 的地址为编码
    TraceChain_df['from_encoded'] = TraceChain_df['from_opcode_id'].map(address_encoding)
    TraceChain_df['to_encoded'] = TraceChain_df['to_opcode_id'].map(address_encoding)
    TraceChain_df = TraceChain_df.drop(columns={'from_opcode_id','to_opcode_id'}).rename(columns={'from_encoded':'from','to_encoded':'to'})

    return TraceChain_df


def extract_transactional_features(df):
    """提取账户的交易活跃度特征，并合并 df 中除 from/to/position 外的交易级特征（按 from 聚合）。"""
    
    # ------------------- 1. 原有活跃度特征 -------------------
    # 获取所有唯一账户 ID
    all_nodes = pd.Series(df['from'].tolist() + df['to'].tolist()).unique()
    
    # 1.1 出度：账户发起交易数量
    out_degree = df['from'].value_counts().rename('out_degree')
    
    # 1.2 入度：账户接收交易数量
    in_degree = df['to'].value_counts().rename('in_degree')
    
    # 1.3 基础特征 DataFrame
    features_df = pd.DataFrame(index=all_nodes)
    features_df = features_df.join(out_degree, how='left').fillna(0)
    features_df = features_df.join(in_degree, how='left').fillna(0)
    
    # 1.4 衍生特征
    features_df['total_tx'] = features_df['out_degree'] + features_df['in_degree']
    features_df['net_flow'] = features_df['out_degree'] - features_df['in_degree']
    
    
    # ------------------- 2. 合并其他交易特征（按 from 聚合） -------------------
    # 排除的列
    exclude_cols = ['from', 'to', 'position']
    # 保留需要合并的列
    extra_cols = [col for col in df.columns if col not in exclude_cols]
    
    if extra_cols:
        # 按 'from' 聚合
        agg_dict = {}
        for col in extra_cols:
            if pd.api.types.is_numeric_dtype(df[col]):
                agg_dict[col] = 'sum'          # 数值列求和（如金额）
            else:
                agg_dict[col] = 'first'        # 非数值列取第一个（可改为 'nunique' / 'count' 等）
        
        # 聚合
        extra_features = df.groupby('from')[extra_cols].agg(agg_dict)
        
        # 添加前缀避免列名冲突（如 amount_sum）
        extra_features = extra_features.add_prefix('from_')
        
        # 合并到 features_df（左连接，缺失补 0 或 NaN）
        features_df = features_df.join(extra_features, how='left')
        
        # 可选：数值列缺失值填 0
        numeric_cols = extra_features.select_dtypes(include='number').columns
        features_df[numeric_cols] = features_df[numeric_cols].fillna(0)
    
    # ------------------- 3. 类型转换 -------------------
    # 确保出度/入度等为整数
    int_cols = ['out_degree', 'in_degree', 'total_tx', 'net_flow']
    features_df[int_cols] = features_df[int_cols].astype(int)
    
    return features_df


def align_node_features(ponzi_contract_data):
    """
    统一所有 node_features 的列（列对齐 + 补0）
    """
    all_feature_dfs = [item['node_features'] for item in ponzi_contract_data.values()]
    
    # 找出所有图中出现过的列
    all_columns = set()
    for df in all_feature_dfs:
        all_columns.update(df.columns)
    all_columns = sorted(list(all_columns))
    
    print(f"统一特征维度: {len(all_columns)} 列 → {all_columns}")
    
    # 对每个图的 node_features 做 reindex + fillna(0)
    for addr, item in ponzi_contract_data.items():
        df = item['node_features']
        df_aligned = df.reindex(columns=all_columns, fill_value=0)
        item['node_features'] = df_aligned.astype('float32')  # 确保类型一致
    
    return ponzi_contract_data, all_columns


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing, global_mean_pool
from torch_geometric.utils import add_self_loops, degree


class EdgeWeightedGCNConv(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')  # 聚合方式为加和
        self.lin = nn.Linear(in_channels, out_channels)
        self.edge_mlp = nn.Sequential(
            nn.Linear(1, out_channels),  # 这里假设每条边有一个标量特征
            nn.ReLU(),
            nn.Linear(out_channels, 1),
            nn.Sigmoid()  # 输出权重在 0~1 之间
        )

    def forward(self, x, edge_index, edge_attr=None):
        # 添加自环
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))

        # 如果没有边特征，就初始化为 1
        if edge_attr is None:
            edge_attr = torch.ones(edge_index.size(1), 1, device=x.device)

        # 线性变换节点特征
        x = self.lin(x)

        # 计算归一化系数
        row, col = edge_index
        deg = degree(row, x.size(0), dtype=x.dtype)
        deg_inv_sqrt = deg.pow(-0.5)
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

        # 计算 learnable edge weights
        edge_weight = self.edge_mlp(edge_attr).view(-1)
        norm = norm * edge_weight  # 加权邻接矩阵

        # 消息传递
        return self.propagate(edge_index, x=x, norm=norm)

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class EdgeWeightedGCN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim,
                 num_layers, dropout):
        super().__init__()
        self.layers = nn.ModuleList()
        self.bns = nn.ModuleList()

        # 输入层
        self.layers.append(EdgeWeightedGCNConv(input_dim, hidden_dim))
        self.bns.append(nn.BatchNorm1d(hidden_dim))

        # 中间层
        for _ in range(num_layers - 2):
            self.layers.append(EdgeWeightedGCNConv(hidden_dim, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))

        # 输出层
        self.layers.append(EdgeWeightedGCNConv(hidden_dim, output_dim))
        self.bns.append(nn.BatchNorm1d(output_dim))

        # 分类头
        self.classifier = nn.Sequential(
            nn.Linear(output_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 2)
        )

        self.dropout = dropout

    def forward(self, x, edge_index, batch, edge_attr=None):
        h = x
        for conv, bn in zip(self.layers, self.bns):
            h = conv(h, edge_index, edge_attr)
            h = bn(h)
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)

        # 图级池化
        graph_emb = global_mean_pool(h, batch)

        # 分类
        logits = self.classifier(graph_emb)

        return logits, graph_emb



import torch
import numpy as np
import networkx as nx
from torch_geometric.data import Data
from torch_geometric.utils import from_networkx


def build_pyg_data_list(ponzi_contract_data):
    """
    输入: ponzi_contract_data 字典
    输出: List[Data]，每个 Data 包含:
        - x: 节点特征
        - edge_index: 边索引
        - edge_attr: 边特征（交易数量）
        - y: 图标签
        - node_ids: 节点索引映射
    """
    data_list = []

    for addr, item in ponzi_contract_data.items():
        trace_graph_df = item['trace_graph_df']
        node_features = item['node_features']
        label = item['label']

        # --- 1. 构建 NetworkX 图 ---
        if trace_graph_df.empty:
            G = nx.DiGraph()
            G.add_node(addr)
        else:
            G = nx.from_pandas_edgelist(
                trace_graph_df,
                source='from',
                target='to',
                edge_attr=True,       # 加上边属性
                create_using=nx.DiGraph()
            )

        # --- 2. 节点特征对齐 ---
        all_nodes = list(G.nodes())
        node_id_map = {node: i for i, node in enumerate(all_nodes)}
        feature_matrix = []
        for node in all_nodes:
            if node in node_features.index:
                feat = node_features.loc[node].values
            else:
                feat = np.zeros(node_features.shape[1])
            feature_matrix.append(feat)
        x = torch.tensor(np.array(feature_matrix), dtype=torch.float)

        # --- 3. 边索引与边特征 ---
        # edge_index: [2, E]
        # edge_attr: [E, edge_dim]
        data_nx = from_networkx(G)

        edge_index = data_nx.edge_index

        # 提取边特征
        if not trace_graph_df.empty and 'value' in trace_graph_df.columns:
            # 取出每条边的 value，按顺序与 edge_index 对齐
            edge_values = trace_graph_df['value'].fillna(0).values
            edge_attr = torch.tensor(edge_values, dtype=torch.float).view(-1, 1)
        else:
            # 如果没有边特征，则使用全 1
            edge_attr = torch.ones(edge_index.size(1), 1, dtype=torch.float)

        # --- 4. 图标签 ---
        y = torch.tensor([label], dtype=torch.long)

        # --- 5. 封装为 Data 对象 ---
        data = Data(
            x=x,
            edge_index=edge_index,
            edge_attr=edge_attr,
            y=y
        )
        data.num_nodes = len(all_nodes)
        data.addr = addr
        data.node_ids = torch.tensor(
            [node_id_map[n] for n in all_nodes],
            dtype=torch.long
        )

        data_list.append(data)

    return data_list


In [5]:
# 目标字典，现在用于存储包含 trace_graph_df, node_features 和 label 的字典
ponzi_contract_data = {} 

# 1. 确保获取的是唯一的地址列表
unique_addresses = list(set(ponzi_contract['address']))

for addr in unique_addresses:
    # 2.1 构造 Trace Graph DataFrame
    # 假设 construct_EthTrace_graph(ponzi_contract, addr) 返回一个 DataFrame
    trace_graph_df = construct_EthTrace_graph(ponzi_contract, addr)
    # 2.2 提取特征
    transactional_features = extract_transactional_features(trace_graph_df)

    # 2.3 合并节点特征
    # 账户ID作为索引。fillna(0) 处理拓扑特征中可能因图太小或隔离节点导致的 NaN
    node_features = transactional_features
    
    # 2.4 获取合约标签
    contract_label = label_value = ponzi_contract[ponzi_contract['address'] == addr]['label'].iloc[0]
    
    # 2.5 存储所有信息到一个结构化的字典中
    ponzi_contract_data[addr] = {
        'trace_graph_df': trace_graph_df, # 原始边列表
        'node_features': node_features,   # 提取的节点特征
        'label': contract_label           # 合约标签 (图级别标签)
    } 

ponzi_contract_data, all_columns = align_node_features(ponzi_contract_data)

统一特征维度: 146 列 → ['from_opcode_ADD', 'from_opcode_ADDMOD', 'from_opcode_ADDRESS', 'from_opcode_AND', 'from_opcode_BALANCE', 'from_opcode_BLOCKHASH', 'from_opcode_BYTE', 'from_opcode_CALL', 'from_opcode_CALLCODE', 'from_opcode_CALLDATACOPY', 'from_opcode_CALLDATALOAD', 'from_opcode_CALLDATASIZE', 'from_opcode_CALLER', 'from_opcode_CALLVALUE', 'from_opcode_CHAINID', 'from_opcode_CODECOPY', 'from_opcode_CODESIZE', 'from_opcode_COINBASE', 'from_opcode_CREATE', 'from_opcode_CREATE2', 'from_opcode_DELEGATECALL', 'from_opcode_DIFFICULTY', 'from_opcode_DIV', 'from_opcode_DUP1', 'from_opcode_DUP10', 'from_opcode_DUP11', 'from_opcode_DUP12', 'from_opcode_DUP13', 'from_opcode_DUP14', 'from_opcode_DUP15', 'from_opcode_DUP16', 'from_opcode_DUP2', 'from_opcode_DUP3', 'from_opcode_DUP4', 'from_opcode_DUP5', 'from_opcode_DUP6', 'from_opcode_DUP7', 'from_opcode_DUP8', 'from_opcode_DUP9', 'from_opcode_EQ', 'from_opcode_EXP', 'from_opcode_EXTCODECOPY', 'from_opcode_EXTCODEHASH', 'from_opcode_EXTCODESIZE',

In [9]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25, reduction='mean'):
        super(FocalLoss, self).__init__()
        # gamma (float): 聚焦参数，用于调整难易样本的损失贡献
        self.gamma = gamma
        # alpha (float/tensor): 类别平衡参数，用于调整正负样本的权重
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, input, target):
        # input: [batch_size, num_classes] (Logits)
        # target: [batch_size] (Class indices)
        
        # 1. 计算标准交叉熵损失
        # reduction='none' 确保返回每个样本的损失值，以便后续计算 (1-p_t)^gamma
        ce_loss = F.cross_entropy(input, target, reduction='none')
        
        # 2. 计算 p_t: 模型对真实类别的预测概率
        # p_t = exp(-ce_loss)
        p_t = torch.exp(-ce_loss)
        
        # 3. 计算 modulating factor: (1 - p_t)^gamma
        focal_term = (1.0 - p_t)**self.gamma
        
        # 4. 计算 alpha_t
        # 注意: 您的任务是二分类 (Ponzi/Non-Ponzi)，假设 Ponzi 是类别 1。
        # target_one_hot = F.one_hot(target, num_classes=input.size(-1)).float()
        # if input.size(-1) == 2:
        #     # 简单二分类 alpha_t: 类别 0 权重 (1-alpha)，类别 1 权重 (alpha)
        #     alpha_t = self.alpha * target_one_hot[:, 1] + (1 - self.alpha) * target_one_hot[:, 0]
        # else:
        #     # 对于多分类或更通用的情况，使用类别索引查找
        #     # 简单的 alpha 实现: 假设 alpha 是一个包含 num_classes 权重的 Tensor
        #     # 如果 alpha 是一个 float，则默认只用于 minority class (index=1)
        #     alpha_t = torch.ones_like(target).float()
        #     alpha_t[target == 1] = self.alpha 
        #     alpha_t[target == 0] = 1 - self.alpha
        
        # 简化版 alpha_t (假设 alpha 是应用于少数类 1 的权重)
        if isinstance(self.alpha, (float, int)):
            alpha_t = torch.ones_like(target).float() * (1 - self.alpha)
            # 假设少数类是 1 (Ponzi)
            alpha_t[target == 1] = self.alpha 
        else:
             # 如果 alpha 是一个权重张量，则可以直接使用 target 索引
             alpha_t = self.alpha[target]

        # 5. 计算 Focal Loss: - alpha_t * (1 - p_t)^gamma * log(p_t)
        # 由于 -log(p_t) = ce_loss，所以 Loss = alpha_t * focal_term * ce_loss
        loss = alpha_t * focal_term * ce_loss
        
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss

In [ ]:
from numpy import average
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score,confusion_matrix
import torch.optim as optim
import torch.nn as nn

# --- 1. 构建数据集 ---
data_list = build_pyg_data_list(ponzi_contract_data)

# --- 2. 划分训练 / 验证 / 测试 ---
temp_data, test_data = train_test_split(data_list, test_size=0.3, random_state=42, stratify=[d.y.item() for d in data_list])
train_data, val_data = train_test_split(temp_data, test_size=0.25, random_state=42, stratify=[d.y.item() for d in temp_data])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

# --- 3. 初始化模型 ---
input_dim = data_list[0].x.shape[1]
model = EdgeWeightedGCN(input_dim=input_dim, hidden_dim=64, output_dim=64, num_layers=3, dropout=0.5)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

# --- 4. 训练循环 ---
def train_epoch():
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        logits, _ = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(logits, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

@torch.no_grad()
def evaluate(loader):
    model.eval()
    preds, trues, probs = [], [], []
    for batch in loader:
        batch = batch.to(device)
        logits, _ = model(batch.x, batch.edge_index, batch.batch)
        prob = F.softmax(logits, dim=1)
        pred = logits.argmax(dim=1)
        
        preds.extend(pred.cpu().numpy())
        trues.extend(batch.y.cpu().numpy())
        probs.extend(prob[:, 1].cpu().numpy())  # Ponzi 概率
    
    acc = accuracy_score(trues, preds)
    # pre = precision_score(trues, preds)
    # recall = recall_score(trues, preds)
    # f1 = f1_score(trues, preds)
    pre = precision_score(trues, preds, average='weighted')
    recall = recall_score(trues, preds, average='weighted')
    f1 = f1_score(trues, preds, average='weighted')
    auc = roc_auc_score(trues, probs)
    cm = confusion_matrix(trues, preds)
    return acc, f1, auc, recall, pre, cm

# --- 5. 训练 ---
best_val_f1 = 0
for epoch in range(1, 201):
    loss = train_epoch()
    val_acc, val_f1, val_auc, val_recall, val_pre, cm = evaluate(val_loader)
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'best_weightgcn_ponzi.pth')

    print(f'Epoch {epoch:03d} | Loss: {loss:.4f} | Val F1: {val_f1:.4f} | AUC: {val_auc:.4f}')

# --- 6. 测试 ---
model.load_state_dict(torch.load('best_weightgcn_ponzi.pth'))
test_acc, test_f1, test_auc, test_recall, test_pre, cm = evaluate(test_loader)
print(f"best_val_f1: {best_val_f1:.4f} |Test Acc: {test_acc:.4f} | AUC: {test_auc:.4f} | pre: {test_pre:.4f} | F1: {test_f1:.4f} | recall: {test_recall:.4f} | cm:{cm}")

Epoch 001 | Loss: 0.2491 | Val F1: 0.9316 | AUC: 0.8270
Epoch 002 | Loss: 0.1387 | Val F1: 0.9316 | AUC: 0.8949
Epoch 003 | Loss: 0.1164 | Val F1: 0.9789 | AUC: 0.9393
Epoch 004 | Loss: 0.0998 | Val F1: 0.9783 | AUC: 0.9448
Epoch 005 | Loss: 0.0884 | Val F1: 0.9742 | AUC: 0.9237
Epoch 006 | Loss: 0.0969 | Val F1: 0.9584 | AUC: 0.9028
Epoch 007 | Loss: 0.0821 | Val F1: 0.9795 | AUC: 0.9383
Epoch 008 | Loss: 0.0851 | Val F1: 0.9795 | AUC: 0.9107
Epoch 009 | Loss: 0.0844 | Val F1: 0.9786 | AUC: 0.9318
Epoch 010 | Loss: 0.0787 | Val F1: 0.9783 | AUC: 0.9409
Epoch 011 | Loss: 0.0831 | Val F1: 0.9819 | AUC: 0.9330
Epoch 012 | Loss: 0.0841 | Val F1: 0.9799 | AUC: 0.9346
Epoch 013 | Loss: 0.0787 | Val F1: 0.9770 | AUC: 0.9306
Epoch 014 | Loss: 0.0761 | Val F1: 0.9764 | AUC: 0.9428
Epoch 015 | Loss: 0.0669 | Val F1: 0.9812 | AUC: 0.9358
Epoch 016 | Loss: 0.0708 | Val F1: 0.9702 | AUC: 0.9428
Epoch 017 | Loss: 0.0772 | Val F1: 0.9732 | AUC: 0.9184
Epoch 018 | Loss: 0.0637 | Val F1: 0.9814 | AUC:

In [7]:
@torch.no_grad()
def evaluate(loader):
    model.eval()
    preds, trues, probs = [], [], []
    for batch in loader:
        batch = batch.to(device)
        logits, _ = model(batch.x, batch.edge_index, batch.batch)
        prob = F.softmax(logits, dim=1)
        pred = logits.argmax(dim=1)
        
        preds.extend(pred.cpu().numpy())
        trues.extend(batch.y.cpu().numpy())
        probs.extend(prob[:, 1].cpu().numpy())  # Ponzi 概率
    
    acc = accuracy_score(trues, preds)
    pre = precision_score(trues, preds)
    recall = recall_score(trues, preds)
    f1 = f1_score(trues, preds)
    # pre = precision_score(trues, preds, average='weighted')
    # recall = recall_score(trues, preds, average='weighted')
    # f1 = f1_score(trues, preds, average='weighted')
    auc = roc_auc_score(trues, probs)
    cm = confusion_matrix(trues, preds)
    return acc, f1, auc, recall, pre, cm


In [8]:
# --- 6. 测试 ---
model.load_state_dict(torch.load('best_weightgcn_ponzi.pth'))
test_acc, test_f1, test_auc, test_recall, test_pre, cm = evaluate(test_loader)
print(f"best_val_f1: {best_val_f1:.4f} |Test Acc: {test_acc:.4f} | AUC: {test_auc:.4f} | pre: {test_pre:.4f} | F1: {test_f1:.4f} | recall: {test_recall:.4f} | cm:{cm}")

best_val_f1: 0.9879 |Test Acc: 0.9861 | AUC: 0.9542 | pre: 0.8795 | F1: 0.8439 | recall: 0.8111 | cm:[[1841   10]
 [  17   73]]
